In [8]:
from Bio import Entrez
import requests

def pubmed_count_requests(drug, disease):
    term = f'"{drug}"[Title/Abstract] AND "{disease}"[Title/Abstract]'
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "retmode": "json",
        "email": "oseicharlotte633@gmail.com" # Replace with your email
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return int(r.json()["esearchresult"]["count"])

count_1 = pubmed_count_requests("dexamethasone", "")
count_2 = pubmed_count_requests("dexamethasone", "Depression")
count_3 = pubmed_count_requests("dexamethasone", "Hypertension")
count_4 = pubmed_count_requests("dexamethasone", "Rheumatoid Arthritis")

KeyError: 'count'

In [20]:
print(f"Number of PubMed articles for dexamethasone and Asthma: {count_1}")
print(f"Number of PubMed articles for dexamethasone and Depression: {count_2}")
print(f"Number of PubMed articles for dexamethasone and Hypertension: {count_3}")
print(f"Number of PubMed articles for dexamethasone and Rheumatoid Arthritis: {count_4}")

Number of PubMed articles for dexamethasone and Asthma: 70190
Number of PubMed articles for dexamethasone and Depression: 2018
Number of PubMed articles for dexamethasone and Hypertension: 1846
Number of PubMed articles for dexamethasone and Rheumatoid Arthritis: 533


In [ ]:
import requests
import json
from xml.etree import ElementTree as ET
from time import sleep

def pmc_get_pmids(drug, disease, max_count=20):
    term = f'"{drug}"[Title/Abstract] AND "{disease}"[Title/Abstract]'
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pmc",
        "term": term,
        "retmode": "json",
        "retmax": max_count,
        "email": "oseifrancis633@gmail.com"
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    return data.get("esearchresult", {}).get("idlist", [])

def pmc_fetch_abstracts(pmids):
    articles = []
    for i in range(0, len(pmids), 20):
        batch = pmids[i:i+20]
        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        params = {
            "db": "pmc",
            "id": ",".join(batch),
            "retmode": "xml",
            "email": "oseifrancis633@gmail.com"
        }

        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        root = ET.fromstring(r.content)

        for article in root.findall(".//article"):
            title_elem = article.find(".//article-title")
            abstract_elem = article.find(".//abstract")

            title = "".join(title_elem.itertext()).strip() if title_elem is not None else "No Title"
            abstract = "".join(abstract_elem.itertext()).strip() if abstract_elem is not None else "No Abstract"
            
            articles.append({
                "title": title,
                "abstract": abstract
            })

        sleep(0.5)

    return articles

# Example usage
pmids = pmc_get_pmids("dexamethasone", "Depression", max_count=10)
results_json = pmc_fetch_abstracts(pmids)

# Output JSON to screen
print(json.dumps(results_json, indent=2))


[
  {
    "title": "Cortisol and the Dexamethasone Suppression Test as a Biomarker for Melancholic Depression: A Narrative Review",
    "abstract": "The dexamethasone suppression test (DST) assesses the functionality of the HPA axis and can be regarded as the first potential biomarker in psychiatry. In 1981, a group of researchers at the University of Michigan published a groundbreaking paper regarding its use for diagnosing melancholic depression, reporting a diagnostic sensitivity of 67% and a specificity of 95%. While this study generated much enthusiasm and high expectations in the field of biological psychiatry, subsequent studies produced equivocal results, leading to the test being rejected by the American Psychiatric Association. The scientific reasons leading to the rise and fall of the DST are assessed in this review, suggestions are provided as to how the original test can be improved, and its potential applications in clinical psychiatry are discussed. An improved, standard